In [9]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [19]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
import multiprocessing as mp
import os
import cv2
import pickle
import torch
import torch.nn as nn
import torchvision
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
import matplotlib.pyplot as plt
from tqdm import tqdm_notebook
from tqdm import tnrange
import numpy as np
from itertools import combinations
import pandas as pd
import random
import itertools
from math import comb
#Check GPU support, please do activate GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def plot_training_progress(losses, accuracies):
    """
    Plots training loss and accuracy over epochs.

    Args:
        losses (list): List of epoch losses.
        accuracies (list): List of epoch accuracies.
    """
    epochs = range(1, len(losses) + 1)

    plt.figure(figsize=(12, 5))

    # Create a plot for the loss
    ax1 = plt.gca()  # Get current axis
    ax1.plot(epochs, losses, label='Loss', color='blue', marker='o')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss', color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')
    ax1.grid()

    # Create a second y-axis for accuracy
    ax2 = ax1.twinx()  # Create a twin axis sharing the same x-axis
    ax2.plot(epochs, accuracies, label='Accuracy', color='orange', marker='o')
    ax2.set_ylabel('Accuracy', color='orange')
    ax2.tick_params(axis='y', labelcolor='orange')

    # Combine legends
    lines, labels = ax1.get_legend_handles_labels()  # Get the loss labels
    lines2, labels2 = ax2.get_legend_handles_labels()  # Get the accuracy labels
    ax1.legend(lines + lines2, labels + labels2)  # Combine both legends

    plt.tight_layout()
    plt.show()
#-----------------------------Read Img Function---------------------------------
#reading of images
def read_img(A):

    datax = []
    datay = []
    C = os.listdir(A)
    for character in C:
        images = os.listdir(A + character + '/')
        c=0
        for img in images:
          image = cv2.resize(cv2.imread(A + character + '/' + img),(84,84))
          datax.append(image)
          datay.append(character)
          c=c+1
          print(c)

    return np.array(datax), np.array(datay)
#-------------------------------------------------------------------------------
#--------------------------------Read Directory---------------------------------
def read_images(base_directory):
    """
    Reads all the alphabets from the base_directory
    Uses multithreading to decrease the reading time drastically
    """
    datax = None
    datay = None
    #pool = mp.Pool(mp.cpu_count())
    r,r1 =read_img(base_directory)
    return r,r1
def Make_Complete_Matrix(datay, n_way):
    # Get unique classes
    Tot_Classes = np.unique(datay)
    n_classes = len(Tot_Classes)

    # Generate all possible combinations of size n_way
    class_combinations = list(itertools.combinations(Tot_Classes, n_way))

    # Total number of combinations
    total_combinations = comb(n_classes, n_way)

    # Uniform probability for each combination
    uniform_comb_prob = 1 / total_combinations

    # Initialize each combination's probability to uniform value
    comb_prob_dict = {comb: uniform_comb_prob for comb in class_combinations}

    # Create a DataFrame from the combination probabilities
    comb_prob_df = pd.DataFrame(
        list(comb_prob_dict.items()), columns=["Combination", "Probability"]
    )

    # Count how often each class appears across combinations
    class_counts = {cls: 0 for cls in Tot_Classes}
    for combi in class_combinations:
        for cls in combi:
            class_counts[cls] += 1

    '''# Compute marginal probability for each class
    marginal_probs = {cls: class_counts[cls] * uniform_comb_prob for cls in Tot_Classes}
    marginal_probs_df = pd.DataFrame(
        list(marginal_probs.items()), columns=["Class", "Marginal_Probability"]
    )'''
    #print("\nCombination Probabilities:\n", comb_prob_df)
    #print("\nMarginal Probabilities:\n", marginal_probs_df)

    return comb_prob_df


#--------------------Choose class with highest similarity-----------------

'''def Affinity_Class_Matrix(Tot_Classes, n_way, comb_prob_df):
    """
    Select the n-way class combination with the highest affinity (probability)
    from a DataFrame of combinations and their probabilities.

    Parameters:
        Tot_Classes (list): List of all class labels.
        n_way (int): Number of classes in a combination.
        comb_prob_df (pd.DataFrame): DataFrame with 'Combination' and 'Probability' columns.

    Returns:
        tuple: Best class combination (as a tuple).
    """

    # Filter combinations of size n_way (just in case extra combinations are present)
    filtered_df = comb_prob_df[comb_prob_df['Combination'].apply(lambda x: len(x) == n_way)]

    # Select the combination with the highest probability
    best_row = filtered_df.loc[filtered_df['Probability'].idxmax()]
    best_combination = best_row['Combination']

    return best_combination'''


def Affinity_Class_Matrix(Tot_Classes, n_way, comb_prob_df, top_k=5):
    """
    Select a random combination from the top-k most similar (highest probability) combinations.

    Parameters:
        Tot_Classes (list): List of all class labels.
        n_way (int): Number of classes in a combination.
        comb_prob_df (pd.DataFrame): DataFrame with 'Combination' and 'Probability' columns.
        top_k (int): Number of top combinations to consider for random selection.

    Returns:
        tuple: Randomly selected class combination (as a tuple) from top-k.
    """

    # Filter only combinations of length n_way
    filtered_df = comb_prob_df[comb_prob_df['Combination'].apply(lambda x: len(x) == n_way)]

    # Sort by Probability in descending order and take top-k
    top_combinations = filtered_df.sort_values(by='Probability', ascending=False).head(top_k)

    # Randomly select one combination from the top-k
    selected_row = top_combinations.sample(n=1).iloc[0]
    selected_combination = selected_row['Combination']

    return selected_combination



def Make_Comb_Class_Affinity_matrix(z_support, n_support, n_way, support_samples_by_class, comb_prob_df, verbose=False):
    """
    Updates combination probabilities in comb_prob_df based on mean pairwise similarity
    among all support samples from each class combination, excluding intra-class similarities
    and self-similarity (diagonal elements).

    Parameters:
        z_support (Tensor): Support set embeddings (not used directly here).
        n_support (int): Number of support samples per class.
        n_way (int): Number of classes per combination (e.g., 3 for 3-way).
        support_samples_by_class (dict): Mapping from class name to tensor of shape (n_support, embedding_dim).
        comb_prob_df (pd.DataFrame): DataFrame with columns ['Combination', 'Probability'].
        verbose (bool): If True, prints debug info for each combination.

    Returns:
        Updated comb_prob_df with refined similarity-based 'Probability' values.
    """

    new_probs = []

    for idx, row in comb_prob_df.iterrows():
        combination = row['Combination']  # e.g., ('DEB', 'LYM', 'MUS')

        # Gather all support samples for this combination
        try:
            combined_samples = torch.cat(
                [support_samples_by_class[cls] for cls in combination], dim=0
            )  # Shape: (n_way * n_support, embed_dim)
        except KeyError as e:
            raise ValueError(f"Missing class in support_samples_by_class: {e}")

        n_total = combined_samples.size(0)

        # Compute full similarity matrix (including intra- and inter-class)
        similarity_matrix = torch.zeros((n_total, n_total)).to(combined_samples.device)
        for i in range(n_total):
            for j in range(n_total):
                dist = torch.norm(combined_samples[i] - combined_samples[j]) ** 2
                similarity_matrix[i, j] = 1 / (1 + dist)

        # Mask intra-class similarities (excluding pairs within the same class)
        intra_class_mask = torch.zeros((n_total, n_total)).to(combined_samples.device)

        # Populate intra_class_mask to be 1 for intra-class pairs
        class_indices = {}
        start_idx = 0
        for cls in combination:
            num_samples = n_support
            class_indices[cls] = (start_idx, start_idx + num_samples)
            start_idx += num_samples

        for i in range(n_total):
            for j in range(n_total):
                # Check if both i and j are in the same class (intra-class)
                for cls, (start, end) in class_indices.items():
                    if start <= i < end and start <= j < end:
                        intra_class_mask[i, j] = 1  # Mark as intra-class

        # Mask out intra-class similarities and diagonal (self-similarity)
        eye = torch.eye(n_total).to(intra_class_mask.device)  # Identity matrix for diagonal masking
        masked_similarity = similarity_matrix * (1 - intra_class_mask) * (1 - eye)
        #print("masked_similarity",masked_similarity)
        # Mean similarity (excluding intra-class pairs and diagonal)
        mean_similarity = masked_similarity.sum() / (masked_similarity != 0).sum()
        new_probs.append(mean_similarity.item())

        if verbose:
            print(f"Combination {combination} -> Mean Similarity: {mean_similarity.item():.4f}")

    # Normalize all similarities to become probabilities
    new_probs = np.array(new_probs)
    new_probs = new_probs / new_probs.sum()

    # Update probabilities in DataFrame
    comb_prob_df['Probability'] = new_probs
    #print("new_probs", new_probs)
    return comb_prob_df

class Flatten(nn.Module):
    def __init__(self):
        super(Flatten, self).__init__()

    def forward(self, x):
        return x.view(x.size(0), -1)

def load_protonet_conv(**kwargs):
    """
    Loads the prototypical network model
    Arg:
        x_dim (tuple): dimension of input image
        hid_dim (int): dimension of hidden layers in conv blocks
        z_dim (int): dimension of embedded image
    Returns:
        Model (Class ProtoNet)
    """
    x_dim = kwargs['x_dim']
    hid_dim = kwargs['hid_dim']
    z_dim = kwargs['z_dim']

    def conv_block(in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=2, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

    encoder = nn.Sequential(
        conv_block(x_dim[0], hid_dim),
        conv_block(hid_dim, hid_dim),
        conv_block(hid_dim, hid_dim),
        conv_block(hid_dim, z_dim),
        Flatten()
    )

    # Initialize weights using Kaiming Normal initialization
    for m in encoder.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):  # If there are linear layers
            nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    return ProtoNet(encoder)
class ProtoNet(nn.Module):
  def __init__(self, encoder):
    """
    Args:
        encoder : CNN encoding the images in sample
        n_way (int): number of classes in a classification task
        n_support (int): number of labeled examples per class in the support set
        n_query (int): number of labeled examples per class in the query set
    """
    super(ProtoNet, self).__init__()
    self.encoder = encoder.to(device)

  def set_forward_loss(self,n_way, n_support, n_query, datax, datay,PR,matrix_df):
    """
    Computes loss, accuracy and output for classification task
    Args:
        sample (torch.Tensor): shape (n_way, n_support+n_query, (dim))
    Returns:
        torch.Tensor: shape(2), loss, accuracy and y_hat
    """
    if PR==1:
      Tot_Classes = np.unique(datay)
      #-------------Instances from all classes----------------------------
      samples_by_class_support = {}
      samples_by_class_query = {}

      for cls in Tot_Classes:
          datax_cls = datax[datay == cls]
          perm = np.random.permutation(datax_cls)
          sample_cls = perm[:(n_support + n_query)]

          # Convert to tensor and permute to (N, C, H, W)
          class_tensor = torch.from_numpy(sample_cls).float().permute(0, 3, 1, 2)

          # Split support and query
          samples_by_class_support[cls] = class_tensor[:n_support].to(device)  # shape: (n_support, C, H, W)
          samples_by_class_query[cls] = class_tensor[n_support:].to(device)    # shape: (n_query, C, H, W)
      #------------------preparation before passing--------------------------------
      x_support = torch.stack([samples_by_class_support[cls].to(device) for cls in Tot_Classes])  # (n_way, n_support, C, H, W)
      x_query = torch.stack([samples_by_class_query[cls].to(device) for cls in Tot_Classes])    # (n_way, n_query, C, H, W)

      # Step 2: Generate target indices (for classification)
      target_inds = torch.arange(0, len(Tot_Classes)).view(-1, 1, 1).expand(-1, n_query, 1).long().to(device)

      # Step 3: Flatten support and query sets for encoding
      x = torch.cat([
          x_support.contiguous().view(len(Tot_Classes) * n_support, *x_support.size()[2:]),  # Flatten support set
          x_query.contiguous().view(len(Tot_Classes) * n_query, *x_query.size()[2:])  # Flatten query set
      ], 0).to(device)

      # Step 4: Encode with the model's encoder
      z = self.encoder.forward(x)
      #----------------------------Updating of Affinitu class pair matrixes-----------------------------------
      z_support = z[:len(Tot_Classes) * n_support]  # The first n_way * n_support latent features
      z_query = z[len(Tot_Classes) * n_support:]    # The next n_way * n_query latent features

      # Step 6: Create a dictionary to store support samples for each class
      support_samples_by_class = {}

      for i, cls in enumerate(Tot_Classes):
          # Extract the corresponding support samples (already encoded as z)
          support_samples_by_class[cls] = z_support[i * n_support:(i + 1) * n_support]

      # Optionally, print the dictionary to check

      matrix_df=Make_Comb_Class_Affinity_matrix(z_support,n_support,n_way,support_samples_by_class,matrix_df)
      #-------------------------------------select classes with samples--------------------------------------------

      K = Affinity_Class_Matrix(Tot_Classes, n_way, matrix_df)
      class_list = list(Tot_Classes)
      selected_indices = [class_list.index(k_cls) for k_cls in K]

      # Recompute target_inds for selected classes
      target_inds = torch.arange(0, len(K)).view(-1, 1, 1).expand(-1, n_query, 1).long().to(device)

      # Filter z_support and z_query
      z_support_selected = torch.cat([z_support[i * n_support:(i + 1) * n_support] for i in selected_indices], dim=0)
      z_query_selected = torch.cat([z_query[i * n_query:(i + 1) * n_query] for i in selected_indices], dim=0)
      #------------------------------------------------------------------------------------------------------------
      # Ensure gradients are tracked
      z_support_selected.requires_grad_(True)
      z_query_selected.requires_grad_(True)

      #----------------------------Making of prototypes---------------------------
      z_proto = z_support_selected.mean(dim=1)
      #-----------------calculationg distance from each prototypes----------------
      dists = euclidean_dist(z_query,z_query_selected)
      #--------------compute probabilities,loss and accuracy----------------------
      log_p_y = F.log_softmax(-dists, dim=1).view(n_way, n_query, -1)
      loss_val = -log_p_y.gather(2, target_inds).squeeze().view(-1).mean()
      _, y_hat = log_p_y.max(2)
      acc_val = torch.eq(y_hat, target_inds.squeeze()).float().mean()
      #print("matrix_df",matrix_df)
      #print("***************Affinity Matrix Classes Selected***************",K)
    elif PR==0 or matrix_df==None:
      sample = []
      #extract random classes
      K = np.random.choice(np.unique(datay), n_way, replace=False)
      #extract samples from each claass
      for cls in K:
        datax_cls = datax[datay == cls]
        perm = np.random.permutation(datax_cls)
        sample_cls = perm[:(n_support+n_query)]
        sample.append(sample_cls)
      sample = np.array(sample)
      sample = torch.from_numpy(sample).float()
      sample = sample.permute(0,1,4,2,3)
      x_support = sample[:, :n_support].to(device)
      x_query = sample[:, n_support:].to(device)
      #target indices are 0 ... n_way-1
      target_inds = torch.arange(0, n_way).view(n_way, 1, 1).expand(n_way, n_query, 1).long()
      target_inds = Variable(target_inds, requires_grad=False)
      target_inds = target_inds.to(device)

      #encode images of the support and the query set
      x = torch.cat([x_support.contiguous().view(n_way * n_support, *x_support.size()[2:]),
                      x_query.contiguous().view(n_way * n_query, *x_query.size()[2:])], 0)

      z = self.encoder.forward(x)
      z_dim = z.size(-1) #usually 64
      # Extract support and query samples
      z_samples = z[:n_way * n_support].view(n_way, n_support, z_dim)
      z_query = z[n_way * n_support:]

      # Ensure gradients are tracked
      z_samples.requires_grad_(True)
      z_query.requires_grad_(True)

      #----------------------------Making of prototypes---------------------------
      z_proto = z_samples.mean(dim=1)
      #-----------------calculationg distance from each prototypes----------------
      dists = euclidean_dist(z_query,z_proto)
      #--------------compute probabilities,loss and accuracy----------------------
      log_p_y = F.log_softmax(-dists, dim=1).view(n_way, n_query, -1)
      loss_val = -log_p_y.gather(2, target_inds).squeeze().view(-1).mean()
      _, y_hat = log_p_y.max(2)
      acc_val = torch.eq(y_hat, target_inds.squeeze()).float().mean()
      #print("***************Random Classes Selected***************",K)
    return loss_val, {
        'loss': loss_val.item(),
        'acc': acc_val.item(),
        'y_hat': y_hat
        }
def euclidean_dist(x, y):
  """
  Computes euclidean distance btw x and y
  Args:
      x (torch.Tensor): shape (n, d). n usually n_way*n_query
      y (torch.Tensor): shape (m, d). m usually n_way
  Returns:
      torch.Tensor: shape(n, m). For each query, the distances to each centroid
  """
  n = x.size(0)
  m = y.size(0)
  d = x.size(1)
  assert d == y.size(1)

  x = x.unsqueeze(1).expand(n, m, d)
  y = y.unsqueeze(0).expand(n, m, d)

  return torch.pow(x - y, 2).sum(2)
def train(model, optimizer, train_x, train_y, n_way, n_support, n_query, max_epoch, epoch_size,Sam_PR):
    """
    Trains the protonet
    Args:
        model
        optimizer
        train_x (np.array): images of training set
        train_y(np.array): labels of training set
        n_way (int): number of classes in a classification task
        n_support (int): number of labeled examples per class in the support set
        n_query (int): number of labeled examples per class in the query set
        max_epoch (int): max epochs to train on
        epoch_size (int): episodes per epoch
    """
    # Divide the learning rate by 2 at each epoch, as suggested in the paper
    scheduler = optim.lr_scheduler.StepLR(optimizer, 1, gamma=0.5, last_epoch=-1)
    epoch = 0  # Epochs done so far
    stop = False  # Status to know when to stop

    # Lists to store loss and accuracy for plotting
    epoch_losses = []
    epoch_accuracies = []
    #----------------Initializing Class Affinity Matrix------------------------------
    matrix_df = Make_Complete_Matrix(train_y,n_way)

    #----------------------------------------------------------------

    while epoch < max_epoch and not stop:
        running_loss = 0.0
        running_acc = 0.0
        #----------------------Sampling probability of Affinity Sampling-----------------------------------------------
        ours_episode_count = int(epoch_size * Sam_PR)
        sampling_strategy = [True] * ours_episode_count + [False] * (epoch_size - ours_episode_count)
        random.shuffle(sampling_strategy)
        #-------------------------------------------------------------------------------------------------------
        for episode in tnrange(epoch_size, desc="Epoch {:d} train".format(epoch + 1)):
            #-------------------------------------Affinity factors------------------------------------
            if sampling_strategy[episode] or episode == 0:
              # Use normal sampling
              PR=0
            elif epoch == 0:
              # Use normal sampling
              PR=0
            else:
              # Use adaptive sampling
              PR=1
            #-----------------------------------------------------------------------------
            optimizer.zero_grad()
            loss, output = model.set_forward_loss(n_way, n_support, n_query, train_x, train_y, PR, matrix_df=matrix_df)

            running_loss += output['loss']
            running_acc += output['acc']
            loss.backward()
            optimizer.step()

        epoch_loss = running_loss / epoch_size
        epoch_acc = running_acc / epoch_size
        print('Epoch {:d} -- Loss: {:.4f} Acc: {:.4f}'.format(epoch + 1, epoch_loss, epoch_acc))

        # Store the values for plotting
        epoch_losses.append(epoch_loss)
        epoch_accuracies.append(epoch_acc)

        epoch += 1
        scheduler.step()

    # Call the plotting function after training
    #plot_training_progress(epoch_losses, epoch_accuracies)

def test(model, test_x, test_y, n_way, n_support, n_query, test_episode):
    """
    Tests the protonet
    Args:
        model: trained model
        test_x (np.array): images of testing set
        test_y (np.array): labels of testing set
        n_way (int): number of classes in a classification task
        n_support (int): number of labeled examples per class in the support set
        n_query (int): number of labeled examples per class in the query set
        test_episode (int): number of episodes to test on
    """
    model.eval()

    loss_list = []
    acc_list = []

    running_loss = 0.0
    running_acc = 0.0
    for episode in range(test_episode):
        loss, output = model.set_forward_loss(n_way, n_support, n_query, test_x, test_y, PR=0, matrix_df=None)


        running_loss += output['loss']
        running_acc += output['acc']

        # Append the current loss and accuracy for later plotting
        loss_list.append(output['loss'])
        acc_list.append(output['acc'])

    avg_loss = running_loss / test_episode
    avg_acc = running_acc / test_episode
    print('Test results -- Loss: {:.4f} Acc: {:.4f}'.format(avg_loss, avg_acc))
    return avg_acc

#----------------------------------make pickle files------------------------------------------------
'''PATH='/content/gdrive/MyDrive/Refined-Pickle-Files/Refined-Pickle-Files/Pickle-84_84/'
Data='Aug_Skin'
trainx, trainy= read_images('/content/gdrive/MyDrive/Datasets/augmented-skin-conditions/train/')
testx, testy = read_images('/content/gdrive/MyDrive/Datasets/augmented-skin-conditions/test/')
with open(PATH+Data+"_trainx.pkl","wb") as f:
  pickle.dump(trainx,f)
with open(PATH+Data+"_trainy.pkl","wb") as f:
  pickle.dump(trainy,f)
with open(PATH+Data+"_testx.pkl","wb") as f:
  pickle.dump(testx,f)
with open(PATH+Data+"_testy.pkl","wb") as f:
  pickle.dump(testy,f)'''
#-----------------------------------------------------------------------------------------------------


#-----------Main function-----------------------------------------------------

n_way = 2
n_support = 1
n_query = 5
epochs_to_run = [20,25,30,35,40]
epoch_size = 20
test_episode = 60
#-------------------------set seeeds-----------------------------
np_seed = 0 # Set the seed for reproducibility
torch_seed = 0
Sam_PR=0.8 #0.6 Blood
Size=84
# Fix all seeds
np.random.seed(np_seed)
torch.manual_seed(torch_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(torch_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Load pickle files
PATH = '/content/gdrive/MyDrive/Refined-Pickle-Files/Refined-Pickle-Files/Pickle-'+str(Size)+'_'+str(Size)+'/'
Data='Aug_Skin'

with open(PATH + Data + "_trainx.pkl", "rb") as f:
    trainx = pickle.load(f)
with open(PATH + Data + "_trainy.pkl", "rb") as f:
    trainy = pickle.load(f)
with open(PATH + Data + "_testx.pkl", "rb") as f:
    testx = pickle.load(f)
with open(PATH + Data + "_testy.pkl", "rb") as f:
    testy = pickle.load(f)

# Function to track the highest accuracy
def get_accuracy(model, testx, testy, n_way, n_support, n_query, test_episode):
    # Assuming this function is available in your framework
    return test(model, testx, testy, n_way, n_support, n_query, test_episode)

best_accuracy = 0
best_epoch = 0

# Training and testing for specified epochs
for epoch in epochs_to_run:
    # Initialize model and optimizer
    model = load_protonet_conv(x_dim=(3, Size, Size), hid_dim=64, z_dim=64)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print(f"-----------------------Training for {epoch} epochs-----------------------")
    train(model, optimizer, trainx, trainy, n_way, n_support, n_query, epoch, epoch_size,Sam_PR)

    print(f"-----------------------Testing after {epoch} epochs-----------------------")
    accuracy = get_accuracy(model, testx, testy, n_way, n_support, n_query, test_episode)

    print(f"Accuracy after {epoch} epochs: {accuracy}")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_epoch = epoch

print(f"\nHighest accuracy: {best_accuracy} achieved at epoch: {best_epoch}")


-----------------------Training for 20 epochs-----------------------


/tmp/ipython-input-1873409866.py:505: TqdmDeprecationWarning: Please use `tqdm.notebook.trange` instead of `tqdm.tnrange`
  for episode in tnrange(epoch_size, desc="Epoch {:d} train".format(epoch + 1)):


Epoch 1 train:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 1 -- Loss: 232.3053 Acc: 0.5800


Epoch 2 train:   0%|          | 0/20 [00:00<?, ?it/s]

...

Epoch 38 -- Loss: 73.8466 Acc: 0.5050


Epoch 39 train:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 39 -- Loss: 81.4832 Acc: 0.4500


Epoch 40 train:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 40 -- Loss: 58.7419 Acc: 0.5100
-----------------------Testing after 40 epochs-----------------------
Test results -- Loss: 36.0256 Acc: 0.5917
Accuracy after 40 epochs: 0.5916666731238365

Highest accuracy: 0.5916666731238365 achieved at epoch: 40
